In [0]:
# Cell 1 - Mount storage and read raw data
storage_account_name = "deprojectstorage1"
storage_account_key = ""
#"YOUR_STORAGE_ACCOUNT_KEY"
container_raw = "raw"
container_processed = "processed"

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

raw_path = f"abfss://{container_raw}@{storage_account_name}.dfs.core.windows.net/US_Accidents_March23.csv"
df = spark.read.csv(raw_path, header=True, inferSchema=True)
print(f"Raw rows: {df.count():,}")

Raw rows: 7,728,394


In [0]:
# Cell 2 - Drop high-null and irrelevant columns
cols_to_drop = [
    "End_Lat", "End_Lng",          # 44% null
    "Wind_Chill(F)",                # 25% null
    "Precipitation(in)",            # 28% null
    "Description",                  # free text, not useful for aggregation
    "Weather_Timestamp"             # redundant with Start_Time
]

df = df.drop(*cols_to_drop)
print(f"Columns after drop: {len(df.columns)}")

Columns after drop: 40


In [0]:
# Cell 3 - Handle null values
from pyspark.sql.functions import col, median

# Fill numeric nulls with median
numeric_cols = ["Temperature(F)", "Humidity(%)", "Visibility(mi)", 
                "Wind_Speed(mph)", "Pressure(in)"]

for c in numeric_cols:
    median_val = df.approxQuantile(c, [0.5], 0.01)[0]
    df = df.fillna({c: median_val})

# Fill string nulls with mode
df = df.fillna({
    "Weather_Condition": "Clear",
    "Wind_Direction": "Calm",
    "Sunrise_Sunset": "Day",
    "Civil_Twilight": "Day",
    "Nautical_Twilight": "Day",
    "Astronomical_Twilight": "Day"
})

# Drop rows where critical fields are null
df = df.dropna(subset=["Start_Time", "State", "City", "Severity"])

print(f"Rows after null handling: {df.count():,}")

Rows after null handling: 7,728,141


In [0]:
# Cell 4 - Deduplicate and add derived columns
from pyspark.sql.functions import year, month, hour, dayofweek, to_timestamp

# Deduplicate on ID
df = df.dropDuplicates(["ID"])

# Cast and derive time columns
df = df.withColumn("Start_Time", to_timestamp("Start_Time")) \
       .withColumn("End_Time", to_timestamp("End_Time")) \
       .withColumn("Year", year("Start_Time")) \
       .withColumn("Month", month("Start_Time")) \
       .withColumn("Hour", hour("Start_Time")) \
       .withColumn("Day_Of_Week", dayofweek("Start_Time"))

print(f"Final clean rows: {df.count():,}")
print(f"Final columns: {len(df.columns)}")

Final clean rows: 7,728,141
Final columns: 44


In [0]:
# Cell 5 - Create aggregate tables
from pyspark.sql.functions import count, avg, round as spark_round

# Aggregate 1: Accidents by State
agg_by_state = df.groupBy("State") \
    .agg(
        count("ID").alias("total_accidents"),
        spark_round(avg("Severity"), 2).alias("avg_severity")
    ).orderBy("total_accidents", ascending=False)

# Aggregate 2: Accidents by Severity
agg_by_severity = df.groupBy("Severity") \
    .agg(count("ID").alias("total_accidents")) \
    .orderBy("Severity")

# Aggregate 3: Accidents by Year and Month
agg_by_time = df.groupBy("Year", "Month") \
    .agg(count("ID").alias("total_accidents")) \
    .orderBy("Year", "Month")

# Aggregate 4: Accidents by Weather Condition
agg_by_weather = df.groupBy("Weather_Condition") \
    .agg(
        count("ID").alias("total_accidents"),
        spark_round(avg("Severity"), 2).alias("avg_severity")
    ).orderBy("total_accidents", ascending=False).limit(20)

# Aggregate 5: Accidents by Hour of Day
agg_by_hour = df.groupBy("Hour") \
    .agg(count("ID").alias("total_accidents")) \
    .orderBy("Hour")

print("All aggregates created!")

All aggregates created!


In [0]:
# Cell 6 - Write cleaned data and aggregates to processed container
processed_base = f"abfss://{container_processed}@{storage_account_name}.dfs.core.windows.net"

# Write cleaned full dataset as Parquet
df.write.mode("overwrite").parquet(f"{processed_base}/cleaned_accidents")

# Write aggregates as Parquet
agg_by_state.write.mode("overwrite").parquet(f"{processed_base}/agg_by_state")
agg_by_severity.write.mode("overwrite").parquet(f"{processed_base}/agg_by_severity")
agg_by_time.write.mode("overwrite").parquet(f"{processed_base}/agg_by_time")
agg_by_weather.write.mode("overwrite").parquet(f"{processed_base}/agg_by_weather")
agg_by_hour.write.mode("overwrite").parquet(f"{processed_base}/agg_by_hour")

print("All data written to processed container!")

All data written to processed container!


In [0]:
# Cell 7 - Verify everything written correctly
display(agg_by_state.limit(10))
display(agg_by_severity)
display(agg_by_hour)

State,total_accidents,avg_severity
CA,1741422,2.17
FL,880159,2.14
TX,582837,2.22
SC,382557,2.11
NY,347932,2.26
NC,338199,2.13
VA,303301,2.28
PA,296620,2.21
MN,192079,2.16
OR,179655,2.11


Severity,total_accidents
1,67363
2,6156764
3,1299324
4,204690


Hour,total_accidents
0,112372
1,97068
2,93226
3,83862
4,159845
5,228147
6,405819
7,587449
8,577557
9,363022


In [0]:
# Cell 8 - Create master aggregate table for Power BI
from pyspark.sql.functions import count

agg_master = df.groupBy(
    "State",
    "Severity",
    "Year",
    "Hour",
    "Weather_Condition"
).agg(
    count("ID").alias("total_accidents")
)

# Write to ADLS processed container
processed_base = f"abfss://{container_processed}@{storage_account_name}.dfs.core.windows.net"
agg_master.write.mode("overwrite").parquet(f"{processed_base}/agg_master/")

print(f"Master agg table rows: {agg_master.count():,}")
print("Done!")

Master agg table rows: 223,059
Done!
